In [1]:
import os
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.multioutput import MultiOutputRegressor
import shap

warnings.filterwarnings('ignore')

print(" SHAP EXPLAINABILITY ANALYSIS")
data_path = os.path.join('data', 'all_months_features.csv')
df_raw = pd.read_csv(data_path)


/Users/bruker/Desktop/Multi-Target Price Forecasting /.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


 SHAP EXPLAINABILITY ANALYSIS


In [2]:
# Re-run data pipeline setup to align features
df = df_raw.copy()
df['month_idx'] = df.groupby('Product_Name').cumcount() + 1
df = df.sort_values(by=['Product_Name', 'month_idx']).reset_index(drop=True)

targets_base = ['Min_Price', 'Avg_Price', 'Max_Price']
targets_next = ['Min_Price_next', 'Avg_Price_next', 'Max_Price_next']

for base_col, next_col in zip(targets_base, targets_next):
    df[next_col] = df.groupby('Product_Name')[base_col].shift(-1)

df_clean = df.dropna(subset=['Avg_Price_next']).copy().reset_index(drop=True)
df_clean['Min_Price_next'] = df_clean['Min_Price_next'].fillna(df_clean['Avg_Price_next'])
df_clean['Max_Price_next'] = df_clean['Max_Price_next'].fillna(df_clean['Avg_Price_next'])

In [3]:
non_feature_cols = [
    'Product_Name', 'Category', 'Unit', 'unit_canonical', 'month_name',
    'bs_year', 'bs_month'
] + targets_base + targets_next + ['Total_Amount']

feature_cols = [col for col in df_clean.columns if col not in non_feature_cols]
X = df_clean[feature_cols].copy().apply(pd.to_numeric, errors='coerce').fillna(0)
y = df_clean[targets_next].copy().fillna(0)



In [4]:
# Time-based Split: Train (<=7), Valid (==8), Test (==9)
train_mask = (df_clean['month_idx'] <= 7).values
valid_mask = (df_clean['month_idx'] == 8).values
test_mask = (df_clean['month_idx'] == 9).values  # Features from Month 9 -> Target is Month 10

X_train, y_train = X[train_mask], y[train_mask]
X_valid, y_valid = X[valid_mask], y[valid_mask]
X_test, y_test = X[test_mask], y[test_mask]

In [5]:
# Fit LightGBM MultiOutput Model
lgb_base = lgb.LGBMRegressor(
    objective='regression_l1', num_leaves=31, learning_rate=0.05,
    n_estimators=600, random_state=42, n_jobs=-1, verbose=-1
)
model = MultiOutputRegressor(lgb_base)
model.fit(X_train, y_train)

os.path.abspath('shap_plots')
os.makedirs('shap_plots', exist_ok=True)

target_names = ['min', 'avg', 'max']
for i, (name, est) in enumerate(zip(target_names, model.estimators_)):
    print(f"\nGenerating SHAP values for target: {name}")
    explainer = shap.TreeExplainer(est)
    shap_values = explainer(X_test)


Generating SHAP values for target: min

Generating SHAP values for target: avg

Generating SHAP values for target: max


In [6]:
# Beeswarm Plot
plt.figure()
shap.plots.beeswarm(shap_values, show=False)
plt.title(f"SHAP Beeswarm Plot - {name.upper()} Price Next Month")
plt.tight_layout()
plt.savefig(f"shap_plots/beeswarm_{name}.png", dpi=300)
plt.close()

In [7]:
# Global Bar Plot
plt.figure()
shap.plots.bar(shap_values, show=False)
plt.title(f"SHAP Global Feature Importance - {name.upper()} Price")
plt.tight_layout()
plt.savefig(f"shap_plots/global_bar_{name}.png", dpi=300)
plt.close()

In [8]:
# Local Waterfall Plot (First sample in test set)
plt.figure()
explainer_avg = shap.TreeExplainer(model.estimators_[1])
sv_single = explainer_avg(X_test)
shap.plots.waterfall(sv_single[0], show=False)
plt.title("SHAP Local Waterfall Plot - Average Price Sample Prediction")
plt.tight_layout()
plt.savefig("shap_plots/local_waterfall_avg.png", dpi=300)
plt.close()

print("\n✓ All SHAP artefacts successfully generated and saved to 'shap_plots/' directory.")


✓ All SHAP artefacts successfully generated and saved to 'shap_plots/' directory.
